# APQ Demo

In [ ]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from dataclasses import dataclass, field
from typing import List

from src.apq import (
    set_seed, load_calibration_dataset, batch_tokens, 
    evaluate_ppl, detect_arch, calibrate_ap_quant_sequential,
    save_calibration_scales, load_calibration_scales,
    quantize_transformer_attn
)
from src.mckp import compute_sensitivity_matrix, solve_mckp


In [ ]:
@dataclass
class Config:
    model_name: str = "openai-community/gpt2"
    target_bits: int = 4
    candidate_bits: List[int] = field(default_factory=lambda: [2, 3, 4, 8, 16])
    dataset_name: str = "garage-bAInd/Open-Platypus"
    calib_samples: int = 256
    eval_samples: int = 256
    chunk_size: int = 128
    max_len_per_example: int = 256
    batch_size: int = 8
    num_steps_per_layer: int = 200
    learning_rate: float = 1e-3
    init_temp: float = 2.0
    lambda_update_every: int = 10
    val_fraction: float = 0.1
    calibration_passes: int = 3
    lambda_samples: int = 8
    seed: int = 42
    scales_file: str = "calibration_scales.pkl"

In [ ]:
cfg = Config()
set_seed(cfg.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.pad_token = tokenizer.eos_token

calib_tokens, eval_tokens = load_calibration_dataset(
    cfg.dataset_name, tokenizer,
    cfg.calib_samples, cfg.eval_samples,
    cfg.chunk_size, cfg.max_len_per_example
)

batched_calib = batch_tokens(calib_tokens, cfg.batch_size)
batched_eval = batch_tokens(eval_tokens, cfg.batch_size)

print(f"Calibration: {len(calib_tokens)} → {len(batched_calib)} batches")
print(f"Evaluation: {len(eval_tokens)} → {len(batched_eval)} batches")

print("\n=== BASELINE ===")
model_fp = AutoModelForCausalLM.from_pretrained(cfg.model_name, attn_implementation="eager").to(device)
model_fp.eval()
fp_ppl = evaluate_ppl(model_fp, batched_eval, device)
print(f"FP32 PPL: {fp_ppl:.2f}")

print("\n=== UNOPTIMIZED 4-BIT ===")
model_unopt = AutoModelForCausalLM.from_pretrained(cfg.model_name, attn_implementation="eager").to(device)
arch, _ = quantize_transformer_attn(model_unopt)
model_unopt.eval()
unopt_ppl = evaluate_ppl(model_unopt, batched_eval, device)
print(f"Unoptimized PPL: {unopt_ppl:.2f}")

print("\n=== SENSITIVITY ANALYSIS ===")
model_sens = AutoModelForCausalLM.from_pretrained(cfg.model_name, attn_implementation="eager").to(device)
sensitivity, lam = compute_sensitivity_matrix(
    model_sens, batched_calib, cfg.candidate_bits,
    temp=cfg.init_temp, device=device, lambda_samples=cfg.lambda_samples
)
del model_sens

n_layers = len(sensitivity)
cost = {b: b for b in cfg.candidate_bits}
budget = cfg.target_bits * n_layers

assignment, _ = solve_mckp(sensitivity, cost, budget, cfg.candidate_bits)
print(f"Lambda: {lam:.4f}")
print(f"Bit assignment: {assignment}")

print("\n=== CALIBRATION ===")
model_calib = AutoModelForCausalLM.from_pretrained(cfg.model_name, attn_implementation="eager").to(device)

model_calib = calibrate_ap_quant_sequential(
    model_calib, batched_calib, device,
    layer_bits=assignment,
    num_steps_per_layer=cfg.num_steps_per_layer,
    lr=cfg.learning_rate,
    init_temp=cfg.init_temp,
    lambda_update_every=cfg.lambda_update_every,
    batch_size=cfg.batch_size,
    val_fraction=cfg.val_fraction,
    calibration_passes=cfg.calibration_passes,
)

save_calibration_scales(model_calib, cfg.scales_file)

calib_ppl = evaluate_ppl(model_calib, batched_eval, device)
print(f"Calibrated PPL: {calib_ppl:.2f}")

print("\n=== SUMMARY ===")
print(f"Baseline:        {fp_ppl:.2f}")
print(f"Unoptimized:     {unopt_ppl:.2f} (Δ = {unopt_ppl - fp_ppl:+.2f})")
print(f"Calibrated:      {calib_ppl:.2f} (Δ = {calib_ppl - fp_ppl:+.2f})")

if calib_ppl < unopt_ppl:
    recovery = (unopt_ppl - calib_ppl) / (unopt_ppl - fp_ppl + 1e-8) * 100
    print(f"Recovery: {recovery:.1f}%")